# 01 — Data audit · Crowdfund Compass (Group 11)
Goal: load the merged Kickstarter table, re-verify the headline facts, and check the known data issues **before** any modelling.

Workflow: 1) Setup → 2) Import → 3) Verify facts → 4) Known issues → 5) Sanity checks → 6) Conclusions.

*AI use: code written with Claude (Fable 5.2), 16.11.2026; reviewed by the team.*

## 1) Setup
Load the libraries. In Colab, mount Google Drive and point `path` to the `merged_v2` folder (built by `merge_v2.py`).

In [ ]:
import glob
import pandas as pd
import numpy as np
pd.set_option('display.width', 200)

# from google.colab import drive; drive.mount('/content/drive')
path = 'drive/My Drive/ks_scrapes/merged_v2'   # folder with kickstarter_merged_v2_part1..8.parquet and presence_part1..4.parquet

## 2) Import the data
The merged table (version 2, rebuilt 16.11.2026) is split into 8 files. We stack them into one table. One row = one Kickstarter project in its latest observed state.

In [ ]:
files = sorted(glob.glob(path + '/kickstarter_merged_v2_part*.parquet'))
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(len(files), 'files |', len(df), 'rows |', df.id.nunique(), 'unique project ids')

# Readable dates (the raw columns are unix seconds, UTC)
df['launch_date'] = pd.to_datetime(df.launched_at, unit='s')
df['deadline_date'] = pd.to_datetime(df.deadline, unit='s')
df['launch_year'] = df.launch_date.dt.year
df.state.value_counts()

## 3) Verify the headline facts
**Finished** = state is `successful` or `failed`. Success rate = successful / finished.

In [ ]:
fin = df[df.state.isin(['successful', 'failed'])].copy()
fin['success'] = (fin.state == 'successful').astype(int)
print('finished projects:', len(fin))
print('success rate overall:', round(fin.success.mean(), 4))
print('success rate 2014-2017:', round(fin[fin.launch_year.between(2014, 2017)].success.mean(), 4))
print('subcategories:', fin.subcategory.nunique(), '| countries:', fin.country.nunique())
print('parent_category values:', fin.parent_category.nunique(), '(15 expected)')
per_creator = fin.groupby('creator_id').size()
print('creators:', len(per_creator), '| with more than one finished project:', (per_creator > 1).sum())
print('prelaunch_activated known:', round(fin.prelaunch_activated.notna().mean(), 4))

## 4) Known issues
### 4.1 Success bias
Success rate and number of finished projects by launch year. Public benchmark (Statista, citing Kickstarter, January 2025): about 651,000 projects launched, 41.98% funded, 376,698 unsuccessful.

In [ ]:
print(fin.groupby('launch_year').success.agg(n='size', success_rate='mean').round(3))

# How much of Kickstarter do we hold? Compare projects that ended before mid-January 2025 with the benchmark.
pre = df[(df.deadline_date < '2025-01-15') & (df.launched_at > 1e9)]
ours_s = (pre.state == 'successful').sum(); ours_f = (pre.state == 'failed').sum()
bench_s = 651000 * 0.4198; bench_f = 376698
print('successes held:', ours_s, '=', round(ours_s / bench_s, 3), 'of benchmark')
print('failures held :', ours_f, '=', round(ours_f / bench_f, 3), 'of benchmark')
print('our success rate:', round(ours_s / (ours_s + ours_f), 3), 'vs benchmark 0.420')

# Spot check: Kickstarter reported 5,314 of 6,646 tabletop projects funded in 2024 (80%)
t = fin[(fin.subcategory == 'Tabletop Games') & (fin.launch_year == 2024)]
print('Tabletop 2024 in our data:', len(t), 'projects,', t.success.sum(), 'funded, rate', round(t.success.mean(), 3))

**Interpretation:** we hold most successful projects but well under half of the failed ones, and the gap is larger in recent years. Raw success rates in this data are too high and must never be presented as Kickstarter's rates.

### 4.2 `has_video` and `prelaunch_activated` by scrape
Scrape files before 2023-12-14 do not contain these fields. Version 2 of the merge leaves them **missing** there (version 1 wrote `False` for video, which was wrong).

In [ ]:
by_scrape = df.groupby('scrape').agg(rows=('id', 'size'), video_known=('has_video', lambda s: s.notna().mean()),
                                    has_video_share=('has_video', 'mean'),
                                    prelaunch_known=('prelaunch_activated', lambda s: s.notna().mean()))
print(by_scrape.round(3))

# Rule: both fields are only KNOWN for rows whose latest scrape is 2023-12-14 or later
fin['presentation_known'] = fin.has_video.notna() & fin.prelaunch_activated.notna()
print(fin.groupby('launch_year').presentation_known.mean().round(3))

# Is 'known' neutral? No: it depends on the outcome, so it must never be a feature
print(fin[fin.launch_year.between(2010, 2022)].groupby(['launch_year', 'presentation_known']).success.mean().unstack().round(3))

**Interpretation:** both fields are known only for projects last listed in 2023-12 or later, and for all launches from 2023 on (one exception). Whether they are known depends on the outcome (e.g. 2012 launches: 82% funded if known, 57% if unknown). Rule: these two fields, and any "missing" flag for them, are used only in a model trained on launches from 2023 on.

### 4.3 Coverage of 2021–2023

In [ ]:
print(df.groupby('scrape').size())   # rows whose LATEST observation is this scrape
print(fin[fin.launch_year.between(2019, 2026)].groupby('launch_year').size())

**Interpretation:** several scrapes are small, but because later scrapes re-list older projects, each launch year 2021–2023 still has 18–23k finished projects.

### 4.4 Exclusions

In [ ]:
print('not finished (dropped from training):', (~df.state.isin(['successful', 'failed'])).sum())
print('impossible launch date (before Kickstarter existed):', (df.launch_date < '2009-04-01').sum(),
      '| of which finished:', (fin.launch_date < '2009-04-01').sum())
last_scrape = pd.Timestamp('2026-11-16')
print('finished but launched in the last 60 days (only short campaigns can have ended):',
      (fin.launch_date > last_scrape - pd.Timedelta(days=60)).sum())

### 4.5 `parent_category` (fixed in version 2)
Older scrapes have no parent name; version 1 copied the subcategory. Version 2 takes the parent from the category slug (`art/painting` → Art).

In [ ]:
print(fin.groupby('parent_category').size())
print('category ids:', fin.cat_id.nunique(), '| subcategory names:', fin.subcategory.nunique(), '(7 names exist under several parents, so use cat_id / cat_slug as the key)')

### 4.6 Where do the failed projects get lost?
The presence table has one row per project and scrape in which it was listed.

In [ ]:
pres = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(path + '/presence_part*.parquet'))], ignore_index=True)
pres['S'] = pd.to_datetime(pres.scrape); pres['D'] = pd.to_datetime(pres.deadline, unit='s')

# (a) Inside one scrape: does the share of failed projects fall with the time since they ended?
q = pres[(pres.scrape == '2025-12-18') & pres.state.isin(['successful', 'failed'])].copy()
q['months_since_end'] = (q.S - q.D).dt.days // 30
t = q[q.months_since_end.between(0, 12)].groupby(['months_since_end', 'state']).size().unstack()
t['failed_share'] = (t.failed / (t.failed + t.successful)).round(3)
print(t)

# (b) Once listed as finished: are failed projects dropped faster from the next scrape?
A = pres[(pres.scrape == '2025-06-12') & pres.state.isin(['successful', 'failed'])]
print(A.assign(kept=A.id.isin(set(pres[pres.scrape == '2025-12-18'].id))).groupby('state').kept.mean().round(3))

# (c) Projects seen LIVE: do they come back with a final state in a later scrape?
A = pres[(pres.scrape == '2025-06-12') & (pres.state == 'live')].copy()
later = pres[(pres.scrape > '2025-06-12') & pres.state.isin(['successful', 'failed', 'canceled', 'suspended'])]
A['seen_final'] = A.id.isin(set(later.id))
A['funded_share'] = A.pledged / A.goal; A['days_left'] = (A.D - A.S).dt.days
A['group'] = np.select([A.funded_share >= 1, (A.funded_share < 0.2) & (A.days_left <= 7)],
                       ['already funded', 'under 20% funded, 7 days or less left'], 'other')
print(A.groupby('group').seen_final.agg(n='size', seen_final='mean').round(3))

**Interpretation:** failed projects are not dropped over time: their share is flat for 12 months (a) and they are kept from scrape to scrape like successful ones (b). They are lost at the moment the campaign ends: an already-funded live campaign reappears as finished in about 91% of cases, an almost certainly failing one in about 68% (c). Many more failures are never listed at all.

## 5) Sanity checks on the pre-launch fields

In [ ]:
fin['goal_usd'] = fin.goal * fin.static_usd_rate
fin['duration_days'] = (fin.deadline - fin.launched_at) / 86400
fin['prep_days'] = (fin.launched_at - fin.created_at) / 86400
print(fin[['goal_usd', 'duration_days', 'prep_days']].describe(percentiles=[.01, .5, .99]).round(1))
print('goal <= 1 USD:', (fin.goal_usd <= 1).sum(), '| goal >= 10m USD:', (fin.goal_usd >= 1e7).sum(),
      '| duration > 92 days:', (fin.duration_days > 92).sum())
print('state contradicts pledged vs goal:', ((fin.success == 1) & (fin.pledged < fin.goal)).sum()
      + ((fin.success == 0) & (fin.pledged >= fin.goal)).sum())
print('missing: blurb', fin.blurb.isna().sum(), '| creator_id', fin.creator_id.isna().sum(), '| city', fin.city.isna().sum())

## 6) Conclusions
1. The headline facts are confirmed. Version 2 of the merge fixes `parent_category` (15 values).
2. Success bias is large: we hold about 84% of all successes but about 41% of all failures.
3. `has_video` and `prelaunch_activated` are unknown for every project last seen before 2023-12, and "known" depends on the outcome: use them only in a model trained on launches from 2023 on.
4. Coverage by launch year is adequate; drop the last 60 days of launches from the test set.
5. Extreme goals and durations are flagged, not deleted.
6. Failed projects are lost when the campaign ends, not afterwards.